# Worked applications: detrending and residual interpretation

Penalized smoothing is useful for descriptive signal extraction in economics, engineering, demography, and actuarial series. These synthetic applications illustrate interpretation without claiming new empirical discovery or forecast accuracy.

In [ ]:
import awesomefigures as af
from matplotlib_inline.backend_inline import set_matplotlib_formats

# Publication styling only; numerical estimates and metrics are unchanged.
af.set_style("elegante", font="DejaVu Sans", use_tex=False,
             palette="okabe-ito", rasterize=False)
set_matplotlib_formats("svg")

from trendestimation._docs_style import apply_documentation_figure_style
apply_documentation_figure_style()


## Economic-style growth trend

Separate a long-run component from transient cyclical noise. Such two-sided smoothing must not be misinterpreted as a real-time forecast or proof of an economic cycle.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from trendestimation import TrendSmoother,select_lambda
rng=np.random.default_rng(302)
n=100;t=np.arange(n)
growth=100+.55*t+.002*t*t
cycle=4*np.sin(2*np.pi*t/20)
observed=growth+cycle+rng.normal(0,1.6,n)
sm=TrendSmoother(n,2);choice=select_lambda(sm,observed,criterion="gcv")
fit=sm.fit(observed,lambda_=choice.lambda_)
fig,ax=plt.subplots(1,2,figsize=(12,4))
ax[0].plot(observed,color="0.8",label="observed");ax[0].plot(fit.trend,label="estimated trend")
ax[0].plot(growth,"k--",label="truth");ax[0].legend()
ax[1].plot(fit.residual,label="residual");ax[1].axhline(0,color="0.4")
ax[1].legend();fig.tight_layout();plt.show()

## Actuarial graduation illustration

Construct a smooth synthetic age-risk curve with heterogeneity in the observed measurements. This unweighted model is an illustration—not a replacement for exposure-weighted mortality-table methods.

In [ ]:
age=np.arange(20,91);true_risk=np.exp(-7+.062*(age-20))
measured=true_risk*np.exp(rng.normal(0,.12,size=age.size))
log_risk=np.log(measured);s=TrendSmoother(len(age),2)
c=select_lambda(s,log_risk,criterion="gcv")
smoothed=np.exp(s.fit(log_risk,lambda_=c.lambda_).trend)
fig,ax=plt.subplots(figsize=(9,4))
ax.semilogy(age,measured,"o",alpha=.35,label="synthetic observed risk")
ax.semilogy(age,true_risk,"k--",label="simulated truth")
ax.semilogy(age,smoothed,label="graduated curve")
ax.set(xlabel="Age",ylabel="Synthetic risk",title="Illustrative risk graduation (log scale)")
ax.legend();plt.show()

## Engineering sensor signal

Smooth a noisy trajectory while displaying the derivative. Differencing order and smoothing $\lambda$ determine how sharply changes are expressed, especially near boundaries.

In [ ]:
t=np.linspace(0,10,90)
truth=2+np.tanh(t-4.8)
signal=truth+rng.normal(0,.16,len(t))
fig,axs=plt.subplots(1,2,figsize=(12,4))
axs[0].plot(t,signal,color="0.8",label="measurement")
axs[0].plot(t,truth,"k--",label="truth")
for d in [1,2,3]:
    model=TrendSmoother(len(t),d)
    lam=model.lambda_for_smoothness(.68)
    fitted=model.fit(signal,lambda_=lam).trend
    axs[0].plot(t,fitted,label=f"d={d}")
    axs[1].plot(t,np.gradient(fitted,t),label=f"d={d}")
axs[0].set(title="Recovered position");axs[1].set(title="Numerical derivative")
for ax in axs:ax.legend()
fig.tight_layout();plt.show()

## Gaussian MLE variance components for the synthetic sensor

A sensor has **measurement noise**, while the latent trajectory has a distinct variability in its centered second differences. These can be estimated by the Gaussian variance-component model using a smoothness penalty selected by marginal MLE. Because the simulated true trajectory is a fixed nonlinear function (not an exact draw from the Gaussian latent-difference model), the resulting variances are **model-dependent estimates**, not guaranteed recovery of the simulation's noise variance. They do not produce calibrated forecast intervals.

In [ ]:
from trendestimation import estimate_variance_components

sensor_solver = TrendSmoother(len(signal), order=2)
sensor_choice = select_lambda(
    sensor_solver, signal, criterion="mle", mu=0.0
)
sensor_variances = estimate_variance_components(
    sensor_solver, signal, lambda_=sensor_choice.lambda_, mu=0.0
)
print(f"Simulated observation-noise variance: {0.16**2:.5f}")
print(f"Estimated observation-noise variance: {sensor_variances.observation_variance:.5f}")
print(f"Estimated latent second-difference variance: {sensor_variances.difference_variance:.5f}")
fig, ax = plt.subplots(figsize=(9, 3.6))
ax.plot(t, signal, "-", alpha=0.4, label="Noisy sensor readings")
ax.plot(t, truth, "k--", label="Fixed synthetic trajectory")
ax.plot(t, sensor_solver.fit(
    signal, lambda_=sensor_choice.lambda_, mu=0.0
).trend, label="Gaussian MLE trend")
ax.set(xlabel="Time", ylabel="Signal", title="Sensor: MLE smoothing")
ax.legend()
plt.show()